# 3. Improve the chatbot, one part at a time

In notebook 1 we built the chatbot from three parts we chose quickly:

| Part | What we used | Alternatives |
|---|---|---|
| **Parse** | pypdf: the characters in reading order | a layout-aware parser (LlamaParse), OCR for scanned pages |
| **Chunk** | fixed 200-word windows, one page at a time | cut at the document's own headings and keep tables whole; other sizes |
| **Search** | keywords (TF-IDF) | meaning (embeddings); both together ("hybrid") |

Notebook 2 gave us a way to score any version. Now we change **one part at a time** and score each version on the
same 24 questions, the way you would compare models in MLflow: same test set, one change per run.

| Setup | Parse | Chunk | Search |
|---|---|---|---|
| 1. baseline | PDF text | fixed | keywords |
| 2. better parsing | **Markdown** | fixed | keywords |
| 3. + chunk by structure | Markdown | **by headings** | keywords |
| 4. + search by meaning | Markdown | by headings | **embeddings** |

In [1]:
import os
import numpy as np
import pandas as pd

import rag
import rag_eval

pd.set_option("display.width", 200, "display.max_colwidth", None, "display.max_rows", 40)   # show full texts
assert os.environ.get("OPENROUTER_API_KEY"), "No key yet: open llm-cds/.env, paste the key after OPENROUTER_API_KEY=, save, restart the kernel"

SETUPS = {
    "1. baseline: PDF text, fixed, keywords": rag.Settings(docs="pdf", chunking="fixed", retriever="tfidf"),
    "2. Markdown, fixed, keywords":           rag.Settings(docs="markdown", chunking="fixed", retriever="tfidf"),
    "3. Markdown, by headings, keywords":     rag.Settings(docs="markdown", chunking="structure", retriever="tfidf"),
    "4. Markdown, by headings, embeddings":   rag.Settings(docs="markdown", chunking="structure", retriever="embed"),
}
results = {}

## 1. The baseline, again

Re-run notebook 2's baseline so all four setups are scored in the same session, by the same judge (about a minute).

In [2]:
name = "1. baseline: PDF text, fixed, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 21:00:04 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:08, Remaining: 03:10]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:09, Remaining: 01:41]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:10, Remaining: 01:13]

...

Evaluating:  92%|█████████▏| 22/24 [Elapsed: 00:24, Remaining: 00:02]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 29%, scorers: 71%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 29%, scorers: 71%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 1. baseline: PDF text, fixed, keywords
  Run ID: de4ef73e499b49cf94df6a91acfaca90

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

1. baseline: PDF text, fixed, keywords: context_relevance 0.94, groundedness 0.91, answer_relevancy 0.85, refused_in_scope 3, refused_out_of_scope 4   (MLflow run de4ef73e)


## 2. Better parsing: Markdown instead of PDF text

Notebook 1 showed one page both ways. The difference matters most for **tables**: in the PDF text, the cells of a
row are scattered, so a passage can hold "Doxycycline" and "300 mg" without saying which age group they belong to.

How much of the guidelines is in tables? Count the Markdown table rows:

In [3]:
for md_file in sorted(rag.DOCS_MD.glob("*.md")):
    lines = md_file.read_text().splitlines()
    table_rows = sum(line.startswith("|") for line in lines)
    print(f"{md_file.stem:35} {table_rows:5} table rows out of {len(lines):6} lines")

ephi_cebs_2021                        177 table rows out of   1144 lines
ephi_cholera_2022                     317 table rows out of   4375 lines
ephi_climate_sensitive_diseases_2022   150 table rows out of   2722 lines
ephi_emergency_nutrition_2022         407 table rows out of   2382 lines
moh_acute_malnutrition_2019          1352 table rows out of   5693 lines
moh_tb_2017                           753 table rows out of   6426 lines


Only the text changes; the chunking (200-word windows per page) and the search stay the same.

In [4]:
name = "2. Markdown, fixed, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 21:00:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:09, Remaining: 03:27]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:09, Remaining: 01:43]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:09, Remaining: 01:08]

...

Evaluating:  92%|█████████▏| 22/24 [Elapsed: 00:25, Remaining: 00:02]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 30%, scorers: 70%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 30%, scorers: 70%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 2. Markdown, fixed, keywords
  Run ID: 8745d8ad8c9547a1beca2f8dc78bcd27

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

2. Markdown, fixed, keywords: context_relevance 0.98, groundedness 1, answer_relevancy 0.86, refused_in_scope 5, refused_out_of_scope 4   (MLflow run 8745d8ad)


## 3. Chunk by structure: cut at the headings, keep tables whole

Fixed windows cut wherever the 200th word falls: in the middle of a table, or between a heading and the text under
it. The Markdown tells us where the document's own sections are (`#`, `##`, ...). **Structure chunking**:

- one passage per section (long sections are split; very short ones are merged with their neighbour);
- every passage starts with its **heading path**, e.g. `4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management >
  Antibiotic Treatment`, so a passage about "300 mg" also says what it is about;
- a table is **never cut**, even when it runs onto the next page (a passage can cover pages 37–38).

The cholera antibiotics page (38) again, now as structure passages:

In [5]:
structure = rag.make_structure_passages(rag.DOCS_MD)
fixed_md = rag.make_passages(rag.markdown_pages(rag.DOCS_MD))
print(f"fixed windows: {len(fixed_md)} passages;  by structure: {len(structure)} passages\n")

on_page_38 = structure[(structure.doc == "ephi_cholera_2022") & (structure.page <= 38) & (structure.end_page >= 38)]
for p in on_page_38.itertuples():
    print(f"--- {rag.page_label(p.page, p.end_page)}, {len(p.text.split())} words\n{p.text[:500]}\n")

fixed windows: 1508 passages;  by structure: 1187 passages

--- pages 37–38, 206 words
4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management > Monitoring of Patients with Severe Dehydration > Cholera with other infectious disease

The occurrence of new, emerging and re-emerging infectious diseases is challenge to human health. Deadly infectious diseases are one of the continues public health problems. Patients may have sign and symptoms unrelated to cholera for which the cause must be determined and appropriate treatment provided. During cholera epidemic, the deadly infectious disea

--- page 38, 207 words
4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management > Monitoring of Patients with Severe Dehydration > Antibiotic Treatment

* Doxycycline — single dose (300 mg for adults; 2–4 mg/kg for children under 12 years of age) is the antibiotic of choice for all patients, including pregnant women.

* If resistance to doxycycline is documented, give azithromycin 1 g or ciprofloxacin 1 g orally 

In [6]:
lengths = pd.DataFrame({"fixed windows": fixed_md.text.str.split().str.len().describe(),
                        "by structure": structure.text.str.split().str.len().describe()}).round()
lengths

,fixed windows,by structure
count,1508.0,1187.0
mean,159.0,189.0
std,53.0,104.0
min,21.0,24.0
25%,120.0,152.0
50%,200.0,182.0
75%,200.0,206.0
max,200.0,2167.0


> **PAUSE (2 min).** The longest structure passage is very long: what is it, probably? (Hint: a table is never
> cut.) What does a 2,000-word passage cost each time it is retrieved?

In [7]:
name = "3. Markdown, by headings, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 21:01:01 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:05, Remaining: 02:04]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:06, Remaining: 00:43]

Evaluating:  21%|██        | 5/24 [Elapsed: 00:07, Remaining: 00:27]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:18, Remaining: 00:02]

Evaluating:  96%|█████████▌| 23/24 [Elapsed: 00:22, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:30, Remaining: 00:00] [predict_fn: 29%, scorers: 71%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:30, Remaining: 00:00] [predict_fn: 29%, scorers: 71%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 3. Markdown, by headings, keywords
  Run ID: b60b1c2fc0cb499aba34df07b614b265

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

3. Markdown, by headings, keywords: context_relevance 1, groundedness 0.97, answer_relevancy 0.85, refused_in_scope 1, refused_out_of_scope 4   (MLflow run b60b1c2f)


## 4. Search by meaning: embeddings

Keyword search only matches **the same words**. A health worker writes "loose stools", the guideline says
"diarrhoea"; "baby" versus "infant"; Amharic versus English.

An **embedding model** turns a text into a list of numbers (here 384) so that texts with similar **meaning** get
similar numbers, whatever the words. We use `BAAI/bge-small-en-v1.5`: small (67 MB), free, runs on a laptop.
Similarity is measured from 0 (unrelated) to 1 (same meaning):

In [8]:
from fastembed import TextEmbedding

embedder = TextEmbedding(rag.EMBED_MODEL, cache_dir=str(rag.INDEX_CACHE / "models"))


def similarity(a, b):
    va, vb = (np.array(v) for v in embedder.embed([a, b]))
    return float(va @ vb / np.linalg.norm(va) / np.linalg.norm(vb))


pairs = [("my baby has loose stools", "infant with diarrhoea"),
         ("my baby has loose stools", "adult with a cough for two weeks"),
         ("how much ORS", "volume of oral rehydration solution"),
         ("how much ORS", "how much does the vaccine cost")]
for a, b in pairs:
    print(f"{similarity(a, b):.2f}   {a!r}  vs  {b!r}")

0.81   'my baby has loose stools'  vs  'infant with diarrhoea'
0.59   'my baby has loose stools'  vs  'adult with a cough for two weeks'
0.58   'how much ORS'  vs  'volume of oral rehydration solution'
0.67   'how much ORS'  vs  'how much does the vaccine cost'


Each passage is embedded once (cached in `index/`); each question is embedded when asked, and the passages
whose numbers are closest win. Compare the two searches on the same question:

In [9]:
question = "My baby has loose stools and is very thirsty. What should I give?"
keywords = rag.build(SETUPS["3. Markdown, by headings, keywords"])
meaning = rag.build(SETUPS["4. Markdown, by headings, embeddings"])
for label, retriever in [("keywords", keywords), ("embeddings", meaning)]:
    hits = retriever.search(question, 5)
    print(f"{label:11}", "; ".join(rag.source_ids(hits)))
    print("            first passage:", hits.text.iloc[0][:200].replace("\n", " "), "...\n")

keywords    moh_acute_malnutrition_2019:p66-67; ephi_emergency_nutrition_2022:p58; moh_acute_malnutrition_2019:p60; moh_tb_2017:p57-58; ephi_cholera_2022:p32
            first passage: 4 Management of Severe Acute Malnutrition (SAM) > 4.2 Stabilisation Centre (SC) for the Management of SAM With Medical Complications > 4.2.4 Feeding Children 6-59 Months > Transition > Criteria to Mov ...

embeddings  moh_acute_malnutrition_2019:p52; ephi_emergency_nutrition_2022:p79; moh_acute_malnutrition_2019:p40; moh_acute_malnutrition_2019:p114-115; moh_acute_malnutrition_2019:p115-116
            first passage: 4 Management of Severe Acute Malnutrition (SAM) > 4.2 Stabilisation Centre (SC) for the Management of SAM With Medical Complications > 4.2.3 Treatment and Care of Children 6-59 Months > 4.2.3.3 Treat  ...



> **Try it.** Change the question (an everyday phrasing, a misspelling, Amharic) and compare. When do keywords
> win? (Hint: exact drug names, codes, numbers.)

In [10]:
name = "4. Markdown, by headings, embeddings"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 21:01:37 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:07, Remaining: 02:44]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:07, Remaining: 01:20]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:08, Remaining: 00:57]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:23, Remaining: 00:03]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 31%, scorers: 69%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 31%, scorers: 69%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 4. Markdown, by headings, embeddings
  Run ID: 0bb9e54a8713497690682234ec4c72b9

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

4. Markdown, by headings, embeddings: context_relevance 0.96, groundedness 1, answer_relevancy 0.83, refused_in_scope 1, refused_out_of_scope 4   (MLflow run 0bb9e54a)


## 5. Compare

The same four summary numbers for every setup:

In [11]:
comparison = rag_eval.compare(results)
comparison

,context_relevance,groundedness,answer_relevancy,refused_in_scope,refused_out_of_scope
"1. baseline: PDF text, fixed, keywords",0.94,0.91,0.85,3.0,4.0
"2. Markdown, fixed, keywords",0.98,1.00,0.86,5.0,4.0
"3. Markdown, by headings, keywords",1.00,0.97,0.85,1.0,4.0
"4. Markdown, by headings, embeddings",0.96,1.00,0.83,1.0,4.0


And question by question: where did each change help, and where did it hurt? (Groundedness and context
relevance; `R` = refused.)

In [12]:
def by_question(metric):
    table = {}
    for name, t in results.items():
        values = t.set_index("id")[metric].astype(float).round(2).astype(object)
        values[t.set_index("id").refused] = "R"
        table[name.split(":")[0].split(",")[0]] = values
    return pd.DataFrame(table)


questions = rag_eval.load_questions().set_index("id")
by_question("context_relevance").join(questions.type)

,1. baseline,2. Markdown,3. Markdown,4. Markdown,type
id,,,,,
chol_01,1.0,1.0,1.0,1.0,answerable
chol_02,1.0,1.0,1.0,1.0,answerable
chol_03,1.0,1.0,1.0,1.0,answerable
chol_04,1.0,1.0,1.0,1.0,answerable
nutr_01,1.0,R,1.0,1.0,answerable
nutr_02,1.0,1.0,1.0,1.0,answerable
nutr_03,1.0,1.0,1.0,1.0,answerable
csd_01,1.0,1.0,1.0,1.0,answerable
csd_02,1.0,R,1.0,1.0,answerable


In [13]:
by_question("groundedness").join(questions.type)

,1. baseline,2. Markdown,3. Markdown,4. Markdown,type
id,,,,,
chol_01,0.5,1.0,1.0,1.0,answerable
chol_02,1.0,1.0,1.0,1.0,answerable
chol_03,1.0,1.0,1.0,1.0,answerable
chol_04,1.0,1.0,1.0,1.0,answerable
nutr_01,0.5,R,0.5,1.0,answerable
nutr_02,1.0,1.0,1.0,1.0,answerable
nutr_03,1.0,1.0,1.0,1.0,answerable
csd_01,1.0,1.0,1.0,1.0,answerable
csd_02,1.0,R,1.0,1.0,answerable


To read any one answer in full (question id and setup name):

In [14]:
def explain(name, question_id, chars=500):
    row = results[name].set_index("id").loc[question_id]
    print(row.question, "\n\nANSWER:", row.answer, "\n")
    print("  ".join(f"{m} {row[m]:.2f}" if pd.notna(row[m]) else f"{m} -" for m in rag_eval.METRICS), row.judge_notes)
    for n, (source, text) in enumerate(zip(row.sources.split("; "), row.passages), 1):
        print(f"\n[{n}] {source}: {text[:chars]}...")


explain("4. Markdown, by headings, embeddings", "sam_01")

A child in the outpatient therapeutic programme still hasn't met the cure criteria. After how long is she classed as a non-responder, and when would a child instead be counted as a defaulter? 

ANSWER: A breastfed infant 0-6 months is classed as a non-responder if they do not reach the SAM discharge criteria after 16 weeks (4 months) in treatment [4]. A child is counted as a defaulter if they are absent for two consecutive days, and this default should be confirmed [4]. 

context_relevance 0.75  groundedness 1.00  answer_relevancy 0.86 

[1] moh_acute_malnutrition_2019:p75: 4 Management of Severe Acute Malnutrition (SAM) > 4.2 Stabilisation Centre (SC) for the Management of SAM With Medical Complications > 4.2.9 Treatment and Care of Infants 0-6 Months of Age

|Category|Definition|
|-|-|
|New admission|New case who meets the admission criteria for SC (see Table 23).|
|Readmission|Re-admitted for treatment in SC due to relapse or a returned defaulter who meets the admission criteria.|
|

In the **MLflow UI** (`mlflow ui` in `llm-cds/`): experiment **guideline-assistant**, tick the four runs, click
**Compare**. The parameters show exactly what changed between runs; the metrics show what it did. In **Traces**,
filter by run to see the same question answered by different setups.

> **Discussion (10 min).**
> 1. Which setup would you put in front of health workers, and why? Which number did you look at first?
> 2. Did any change make some questions **worse**? Would you have noticed from the averages alone?
> 3. Differences of 0.02 or 0.03 on 20 questions: real, or noise? (Run one setup twice and see how much it moves.)
> 4. What did each change cost? The LlamaParse step is paid, once; embeddings take minutes on a laptop, once;
>    longer passages cost tokens on every question.
> 5. What would you record so that in six months someone can tell **which version** answered a given question?

## Exercises

Each is one changed setting and one `evaluate_setup` call (about a minute):

1. **Number of passages.** `k=3` and `k=10` with the best setup. More context: better or worse? Cost (tokens)?
2. **Chunk size.** Fixed windows of 100 or 400 words (`chunk_words=100, overlap_words=20`).
3. **A smaller model.** `model="qwen/qwen3.5-9b"`: how much worse, for how much cheaper and faster?
4. **The prompt.** Add a new version to `rag.PROMPTS` (for example: "If the passages answer only part of the
   question, say which part is missing") and set `prompt_version=` to it. Does groundedness change? The refusals?
5. **Hybrid search** (harder). Write a retriever that takes the top passages from both keywords and embeddings.

```python
name = "5. k=10"
results[name] = rag_eval.evaluate_setup(rag.Settings(docs="markdown", chunking="structure", retriever="embed", k=10), name)
rag_eval.compare(results)
```